come lavora un layer embedding

In [ ]:
Embedding(input_dim=vocab_size, output_dim=d_model, mask_zero=True) oken ID 0 verrà trattato come padding

In [ ]:
import numpy as np

# 1. Inizializziamo una matrice di embedding (normalmente il modello la inizializza random)
embedding_matrix = np.array([
    [0.1, 0.2, 0.3],   # Embedding per token ID 0
    [0.4, 0.5, 0.6],   # Embedding per token ID 1
    [0.7, 0.8, 0.9],   # Embedding per token ID 2
    [1.0, 1.1, 1.2]    # Embedding per token ID 3
])

print("Matrice degli embedding:\n", embedding_matrix)

# 2. Input: una sequenza di token IDs (esempio: 0, 2, 3)
input_ids = np.array([0, 2, 3])

# 3. Operazione di lookup (simulazione di quello che fa un layer Embedding)
output_embeddings = embedding_matrix[input_ids]

print("\nInput IDs:", input_ids)
print("\nOutput Embedding (vettori corrispondenti):\n", output_embeddings)

Il layer Embedding di Keras è il ponte tra gli ID interi dei token e i vettori numerici su cui il modello può lavorare. Trasforma ogni token ID in un vettore denso, e questi vettori vengono aggiornati durante l’addestramento. È così che il modello impara una rappresentazione numerica significativa per ogni token del vocabolario.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, LayerNormalization, MultiHeadAttention, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# ===== 1. Mini dataset =====
corpus = [
    [1, 2, 3, 4, 5],
    [1, 3, 4, 2, 5],
    [2, 3, 1, 5, 4],
]

vocab_size = 10  # Numero totale di token nel vocabolario
max_seq_len = 5  # Lunghezza delle sequenze

x_train = np.array([seq[:-1] for seq in corpus])  # Input: prime 4 parole
y_train = np.array([seq[1:] for seq in corpus])  # Target: le stesse, shiftate di 1

# ===== 2. Positional Encoding semplice =====
def positional_encoding(length, depth):
    depth = depth / 2
    positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
    depths = np.arange(depth)[np.newaxis, :] / depth # (1, depth)

    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates

    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)],
        axis=-1
    )
    return tf.cast(pos_encoding, dtype=tf.float32)

# ===== 3. Transformer Decoder Layer semplice =====
class SimpleTransformerDecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(d_model),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, x, training, look_ahead_mask=None):
        attn_output = self.mha(x, x, x, attention_mask=look_ahead_mask)
        out1 = self.layernorm1(x + self.dropout1(attn_output, training=training))

        ffn_output = self.ffn(out1)
        out2 = self.layernorm2(out1 + self.dropout2(ffn_output, training=training))
        return out2

# ===== 4. Modello completo Decoder-only =====
class MiniTransformerDecoder(tf.keras.Model):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim, max_seq_len):
        super().__init__()
        self.embedding = Embedding(vocab_size, d_model)
        self.pos_encoding = positional_encoding(max_seq_len, d_model)
        self.decoder_block = SimpleTransformerDecoderBlock(d_model, num_heads, ff_dim)
        self.final_layer = Dense(vocab_size)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x) + self.pos_encoding[:seq_len, :]
        x = self.decoder_block(x, training)
        logits = self.final_layer(x)
        return logits

# ===== 5. Istanziamento e training =====
d_model = 16
num_heads = 2
ff_dim = 64

model = MiniTransformerDecoder(vocab_size, d_model, num_heads, ff_dim, max_seq_len)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=10, batch_size=2)

Epoch 1/10


ValueError: Exception encountered when calling MiniTransformerDecoder.call().

[1mOnly input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: True (of type <class 'bool'>)[0m

Arguments received by MiniTransformerDecoder.call():
  • x=tf.Tensor(shape=(None, 4), dtype=int64)
  • training=True